# GenAI Project — Local PDF RAG Assistant

An end-to-end **LLM + LangChain + RAG** project that runs locally in Jupyter Notebook.

**What it does:** Upload a PDF → extract text → split into chunks → create embeddings → store in FAISS → retrieve relevant chunks → ask an OpenAI LLM to answer using the retrieved context.

**Skills demonstrated:** Python, LangChain, LLM APIs, prompt engineering, embeddings, vector databases/FAISS, semantic search, RAG, document processing, environment variables/API-key handling.

## 1. Install dependencies

Run this cell once in your local environment.

In [ ]:
%pip install -U langchain langchain-openai langchain-community langchain-text-splitters pypdf faiss-cpu python-dotenv

## 2. Imports and API key

The key is entered securely with `getpass`, so it is not displayed in the notebook output.

In [ ]:
import os
import getpass
from pathlib import Path

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter your OpenAI API key: ')

print('API key loaded successfully.')

## 3. Choose your PDF

Put a PDF in the same folder as this notebook and change `PDF_PATH` if necessary.

In [ ]:
PDF_PATH = 'your_document.pdf'  # Example: 'company_annual_report.pdf'

if not Path(PDF_PATH).exists():
    raise FileNotFoundError(
        f'PDF not found: {PDF_PATH}\n'
        'Put your PDF beside this notebook and update PDF_PATH.'
    )

print(f'Using: {PDF_PATH}')

## 4. Load the PDF

Each PDF page becomes a LangChain `Document`.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

print(f'Pages loaded: {len(documents)}')
print('\nFirst page preview:\n')
print(documents[0].page_content[:1000])

## 5. Split the document into chunks

Recursive splitting is a good general-purpose strategy for keeping related text together.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

chunks = text_splitter.split_documents(documents)

print(f'Total chunks: {len(chunks)}')
print('\nExample chunk:\n')
print(chunks[0].page_content[:1200])

## 6. Create embeddings and FAISS vector store

The embedding model converts chunks into vectors. FAISS then lets us retrieve semantically similar chunks.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={'k': 4})

print('FAISS vector store created successfully.')

## 7. Initialize the LLM

You can change the model name if your OpenAI account has access to another model.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model='gpt-5-mini',
    temperature=0,
)

print('LLM initialized.')

## 8. Build the RAG prompt and chain

The LLM is instructed to answer from retrieved document context and avoid inventing facts.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

prompt = ChatPromptTemplate.from_messages([
    ('system', '''You are a helpful document assistant.
Answer the user's question using ONLY the supplied document context.
If the answer is not present in the context, clearly say that the document does not provide enough information.
Do not invent facts.
Give concise but useful answers.

Context:
{context}'''),
    ('human', '{input}')
])

document_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, document_chain)

print('RAG chain created successfully.')

## 9. Ask your first question

In [ ]:
question = 'Summarize the main points of this document.'
response = rag_chain.invoke({'input': question})

print(response['answer'])

## 10. Interactive local chatbot

Type `exit` to stop.

In [ ]:
print('PDF RAG Assistant — type exit to stop.\n')

while True:
    question = input('You: ').strip()
    if question.lower() in {'exit', 'quit', 'q'}:
        print('Goodbye!')
        break
    if not question:
        continue

    result = rag_chain.invoke({'input': question})
    print(f'\nAssistant: {result["answer"]}\n')

    print('Sources:')
    for doc in result['context']:
        page = doc.metadata.get('page', 'unknown')
        print(f'  - page {page + 1 if isinstance(page, int) else page}')
    print()

## 11. Optional: inspect retrieved chunks

This helps you understand what the RAG system actually sent to the LLM.

In [ ]:
test_question = 'What are the most important topics discussed in the document?'
retrieved_docs = retriever.invoke(test_question)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f'--- Retrieved chunk {i} ---')
    print('Page:', doc.metadata.get('page', 'unknown'))
    print(doc.page_content[:800])
    print()

## 12. Optional: save the FAISS index locally

This lets you reuse the vector index later instead of embedding the PDF every time. Only load indexes you created/trust.

In [ ]:
INDEX_DIR = 'faiss_index'
vector_store.save_local(INDEX_DIR)
print(f'FAISS index saved to: {INDEX_DIR}/')

## Project architecture

PDF → PyPDFLoader → Documents → RecursiveCharacterTextSplitter → Chunks → OpenAI Embeddings → FAISS → Retriever → LangChain RAG Chain → OpenAI LLM → Answer + Source pages

### Resume-ready project title
**Intelligent PDF RAG Assistant using LangChain, OpenAI LLM & FAISS**

### Resume bullets
- Built an end-to-end Retrieval-Augmented Generation (RAG) assistant using LangChain and an OpenAI chat model to answer questions from user-provided PDF documents.
- Implemented PDF ingestion, recursive text chunking, OpenAI embeddings, FAISS semantic retrieval, prompt-controlled generation, and source-page reporting.
- Added a local interactive chatbot workflow and persisted the FAISS index for reusable document search.

### Important security note
Never commit your real API key to GitHub. Prefer environment variables or a `.env` file excluded by `.gitignore`.